In [11]:
import pandas as pd
import re

****
FUSION DATASET
****

In [ ]:
# 1. Chargement des fichiers
eval_df = pd.read_csv("../data/extrait_eval.csv")
sirh_df = pd.read_csv("../data/extrait_sirh.csv")
sondage_df = pd.read_csv("../data/extrait_sondage.csv")

# 2. Fonction de nettoyage pour extraire uniquement le chiffre des identifiants
def extract_numeric_id(val):
    if pd.isna(val):
        return None
    # Utilise une regex pour extraire tous les chiffres trouvés dans la chaîne
    numbers = re.findall(r'\d+', str(val))
    return int(numbers[0]) if numbers else None

# 3. Application du nettoyage sur les colonnes clés
# On s'assure que toutes les colonnes de jointure sont au format numérique
eval_df['id_user'] = eval_df['eval_number'].apply(extract_numeric_id)
sirh_df['id_user'] = sirh_df['id_employee'].apply(extract_numeric_id)
sondage_df['id_user'] = sondage_df['code_sondage'].apply(extract_numeric_id)

# 4. Fusion des datasets
# On utilise une fusion 'outer' pour ne perdre aucune donnée si un employé manque dans une source
# Tu peux passer en 'inner' si tu veux seulement les employés présents dans les 3 fichiers
merged_df = pd.merge(sirh_df, eval_df, on='id_user', how='outer')
merged_df = pd.merge(merged_df, sondage_df, on='id_user', how='outer')

# 5. Nettoyage final : suppression de la colonne clé temporaire et des doublons
merged_df = merged_df.drop(columns=['eval_number', 'id_employee', 'code_sondage'])

# 6. Sauvegarde en Parquet
output_path = "../data/merged_hr_data.parquet"
merged_df.to_parquet(output_path, index=False)

print(f"✅ Fusion réussie ! Dataset final sauvegardé sous : {output_path}")
print(f"Dimensions du nouveau dataset : {merged_df.shape}")

✅ Fusion réussie ! Dataset final sauvegardé sous : ../data/merged_hr_data.parquet
Dimensions du nouveau dataset : (1470, 32)


In [18]:
df = pd.read_parquet("../data/merged_hr_data.parquet")

****
ALL FEATURE
****

In [20]:
def analyser_structure(df, nom_fichier):
    """
    Analyse un dataframe déjà chargé.
    """
    print(f"\n--- Analyse de : {nom_fichier} ---")
    print(f"Nombre total de colonnes : {len(df.columns)}")
    print("\nListe des colonnes :")
    
    for i, col in enumerate(df.columns, 1):
        type_col = df[col].dtype
        print(f"{i:02d} | {col:<30} | Type : {type_col}")

# --- Exécution ---
if __name__ == "__main__":
    # 1. Chargement direct du fichier
    chemin = "../data/merged_hr_data.parquet"
    df = pd.read_parquet(chemin)
    
    # 2. Appel de la fonction avec le dataframe chargé
    analyser_structure(df, "merged_hr_data.parquet")


--- Analyse de : merged_hr_data.parquet ---
Nombre total de colonnes : 32

Liste des colonnes :
01 | age                            | Type : int64
02 | genre                          | Type : str
03 | revenu_mensuel                 | Type : int64
04 | statut_marital                 | Type : str
05 | departement                    | Type : str
06 | poste                          | Type : str
07 | nombre_experiences_precedentes | Type : int64
08 | nombre_heures_travailless      | Type : int64
09 | annee_experience_totale        | Type : int64
10 | annees_dans_l_entreprise       | Type : int64
11 | annees_dans_le_poste_actuel    | Type : int64
12 | id_user                        | Type : int64
13 | satisfaction_employee_environnement | Type : int64
14 | note_evaluation_precedente     | Type : int64
15 | niveau_hierarchique_poste      | Type : int64
16 | satisfaction_employee_nature_travail | Type : int64
17 | satisfaction_employee_equipe   | Type : int64
18 | satisfaction_employee_equili

****
Répartition des valeur entre Partie & Rester
****

In [21]:
# 2. Nettoyage spécifique : augementation_salaire_precedente
# On retire le '%' et on convertit en numérique
df['augementation_salaire_precedente'] = (
    df['augementation_salaire_precedente']
    .astype(str)
    .str.replace('%', '', regex=False)
    .apply(pd.to_numeric, errors='coerce')
)

# 3. Préparation des colonnes
cols_a_exclure = [
    'nombre_heures_travailless', 
    'nombre_employee_sous_responsabilite', 
    'ayant_enfants',
    'id_user'
]
df_analyse = df.drop(columns=cols_a_exclure, errors='ignore')
target = 'a_quitte_l_entreprise'

# Identification des colonnes
colonnes_numeriques = df_analyse.select_dtypes(include=['number']).columns.tolist()
if target in colonnes_numeriques:
    colonnes_numeriques.remove(target)

colonnes_categoriel = df_analyse.select_dtypes(include=['object', 'category']).columns.tolist()
if target in colonnes_categoriel:
    colonnes_categoriel.remove(target)

# 4. Analyse Numérique (Moyennes)
print("--- Analyse Moyennes : Variables Numériques (incluant augmentation) ---")
stats_num = df_analyse.groupby(target)[colonnes_numeriques].mean().T
print(stats_num)

# 5. Analyse Catégorielle (Double Pourcentage)
# Total des "Oui" (ceux qui ont quitté) pour le calcul du 1er indicateur
total_oui = df_analyse[df_analyse[target] == 'Oui'].shape[0]

print("\n--- Analyse Répartition : Variables Catégorielles ---")
for col in colonnes_categoriel:
    # Tableau croisé des effectifs
    ct = pd.crosstab(df_analyse[col], df_analyse[target])
    
    # 1. % de Oui dans la catégorie par rapport au total des Oui
    # (Ex: Parmi tous les départs, quelle part sont des Célibataires ?)
    pct_dans_total_oui = (ct['Oui'] / total_oui) * 100
    
    # 2. % de Oui par rapport au total de la catégorie
    # (Ex: Parmi tous les Célibataires, quel % est parti ?)
    pct_par_categorie = (ct['Oui'] / ct.sum(axis=1)) * 100
    
    # Fusion pour affichage
    resultat = pd.DataFrame({
        '% dans total Oui': pct_dans_total_oui,
        '% de départ par Catégorie': pct_par_categorie
    }).round(2)
    
    print(f"\nRépartition pour : {col}")
    print(resultat)

--- Analyse Moyennes : Variables Numériques (incluant augmentation) ---
a_quitte_l_entreprise                              Non          Oui
age                                          37.561233    33.607595
revenu_mensuel                             6832.739659  4787.092827
nombre_experiences_precedentes                2.645580     2.940928
annee_experience_totale                      11.862936     8.244726
annees_dans_l_entreprise                      7.369019     5.130802
annees_dans_le_poste_actuel                   4.484185     2.902954
satisfaction_employee_environnement           2.771290     2.464135
note_evaluation_precedente                    2.770479     2.518987
niveau_hierarchique_poste                     2.145985     1.637131
satisfaction_employee_nature_travail          2.778589     2.468354
satisfaction_employee_equipe                  2.733982     2.599156
satisfaction_employee_equilibre_pro_perso     2.781022     2.658228
note_evaluation_actuelle                    

C:\Users\ThibautSénéchal\AppData\Local\Temp\ipykernel_27900\865560620.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_categoriel = df_analyse.select_dtypes(include=['object', 'category']).columns.tolist()
